<a href="https://colab.research.google.com/github/akgupta63/CODSOFT_TASK2/blob/main/CREDIT_CARD_FRAUD_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Extracting and Loading the Dataset
We will unzip the `fraudTest.csv.zip` file to access the CSV, and then load it into a pandas DataFrame.

In [1]:
import zipfile
import pandas as pd

# Unzip the file
zip_path = '/content/fraudTest.csv.zip'
extract_dir = '/content/'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Load dataset
# The extracted file should have a similar name to the zip, typically 'fraudTest.csv'
csv_path = '/content/fraudTest.csv'
df = pd.read_csv(csv_path)

# Display basic information and shape
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (555719, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


In [2]:
# Check class distribution to understand the imbalance
if 'is_fraud' in df.columns:
    print("Class distribution of 'is_fraud':")
    display(df['is_fraud'].value_counts(normalize=True))
else:
    print("Columns present in the dataset:")
    print(df.columns)

Class distribution of 'is_fraud':


,proportion
is_fraud,
0,0.99614
1,0.00386


### 2. Feature Engineering and Preprocessing
We will extract relevant features like transaction hour, day of the week, and calculate the customer's age at the time of transaction. We will also perform target encoding or one-hot encoding on important categorical variables (like `category` and `gender`) and normalize numerical inputs.

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert datetime columns to datetime objects
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

# Feature engineering
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
df['trans_day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

# Define feature columns
num_features = ['amt', 'lat', 'long', 'city_pop', 'unix_time', 'merch_lat', 'merch_long', 'trans_hour', 'trans_day_of_week', 'age']
cat_features = ['category', 'gender']

# One-hot encode the selected categorical columns
X = pd.get_dummies(df[num_features + cat_features], columns=cat_features, drop_first=True)
y = df['is_fraud']

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Scale only the original numerical features
X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

Training set shape: (389003, 24)
Test set shape: (166716, 24)


### 3. Model Training & Evaluation
Now we will train and compare three classifiers:
1. **Logistic Regression** (with class weight adjustment)
2. **Decision Tree Classifier** (with class weight adjustment)
3. **Random Forest Classifier** (with class weight adjustment, optimized for speed)

We will evaluate using metrics like Precision, Recall, F1-Score, and AUC-ROC, which are crucial for highly imbalanced datasets.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

# Dictionary to hold models and their performances
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    # We limit n_estimators and max_depth for speed given the large dataset size
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
}

for name, model in models.items():
    print(f"=== Training {name} ===")
    model.fit(X_train_scaled, y_train)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

    # Evaluate
    print(f"{name} Evaluation:")
    print(classification_report(y_test, y_pred))
    print(f"AUC-ROC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("="*50)

=== Training Logistic Regression ===
Logistic Regression Evaluation:
              precision    recall  f1-score   support

           0       1.00      0.89      0.94    166072
           1       0.03      0.76      0.05       644

    accuracy                           0.88    166716
   macro avg       0.51      0.82      0.49    166716
weighted avg       1.00      0.88      0.94    166716

AUC-ROC Score: 0.9165
Confusion Matrix:
[[146993  19079]
 [   152    492]]
=== Training Decision Tree ===
Decision Tree Evaluation:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99    166072
           1       0.12      0.94      0.21       644

    accuracy                           0.97    166716
   macro avg       0.56      0.95      0.60    166716
weighted avg       1.00      0.97      0.98    166716

AUC-ROC Score: 0.9656
Confusion Matrix:
[[161440   4632]
 [    40    604]]
=== Training Random Forest ===
Random Forest Evaluation:
             